# 07 章 / 第 1 节 · 手写迷你 FlashAttention2(Triton)

## 学习目标
- 理解 FlashAttention 三件套:**tiling + online softmax + 避免实例化 N×N 矩阵**
- 用 **Triton** 手写一个 forward-only mini FA2 kernel
- 与 PyTorch `F.scaled_dot_product_attention` 做 `torch.allclose(rtol=1e-2)` 数值校验
- 出 **naive / SDPA / mini-Triton** 的吞吐对比表

## 对应八股
`liguodongiot/llm-action`:**`llm-interview/llm-algo.md`**(FlashAttention 节)+ `ai-infra/`

## ⚠️ 运行要求
**Linux + NVIDIA GPU**(Triton 官方只支持 Linux,Windows 装不上)。
本节也提供 `naive_attention` 纯 PyTorch 参考实现 —— Windows 用户至少能跑这部分看数值。


## 1. 概念背景

### 1.1 为什么标准 attention 慢?

```
Q ∈ (N, d), K ∈ (N, d), V ∈ (N, d)
S = Q @ K.T          # (N, N)  ← 这一步把 N×N 中间矩阵写到 HBM
P = softmax(S)       # (N, N)
O = P @ V            # (N, d)
```

`N=2048, d=64` 时 `S` 是 16MB(bf16),完全装不进 SM 的 **SRAM**(每 SM 100-200KB),
所以 GPU 不得不:**算完 `S` → 写 HBM → 读回 → softmax → 写 HBM → 再读回 → 乘 V**。
HBM ↔ SM 的搬运是瓶颈,不是计算。

### 1.2 FlashAttention 怎么解决?

**Tiling**:把 Q 切块 `BLOCK_M`,K/V 切块 `BLOCK_N`,**逐块塞进 SRAM 算完再写回**,
中间矩阵从未被实例化。

**Online softmax** 是关键:
传统 softmax 需要全行 max 和 sum,**逐块算时无法预知未来的块**。
Online softmax 维护 `(m_i, l_i)`(running max / running sum),每来一块就**校正**:

```
m_new = max(m_old, m_block)
α = exp(m_old - m_new)           # 旧值的校正因子
l_new = α * l_old + sum(exp(s_block - m_new))
O_new = α * O_old + exp(s_block - m_new) @ V_block
```

数值上等价于普通 softmax,但不需要全行可见。

### 1.3 FA v2 比 v1 改了什么

- **更少非 matmul FLOPs**:把 softmax rescale 推到外面(只在最后除一次 l)
- **沿序列维并行**:v1 只沿 batch 维并行,decode 时 batch=1 就废了;v2 沿 seq 维切,GPU 占用率拉满
- 本节实现的是 v2 思路


## 2. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check()

# 额外检测 Triton(Windows 上会 WARN,继续也能读)
try:
    import triton; print(f"\n[OK]   triton             {triton.__version__}")
    HAS_TRITON = True
except ImportError as e:
    print(f"\n[WARN] triton             未安装/Windows 不支持({e})")
    HAS_TRITON = False


## 3. Naive Reference 实现(PyTorch,用于数值校验)


In [ ]:
import math
import torch
import torch.nn.functional as F


def naive_attention(q, k, v, causal=True):
    """教学参考:实例化 N×N 中间矩阵。q/k/v: (B, H, N, D)"""
    d = q.shape[-1]
    s = (q @ k.transpose(-2, -1)) / math.sqrt(d)   # (B, H, N, N)
    if causal:
        N = s.shape[-1]
        mask = torch.triu(torch.ones(N, N, device=s.device, dtype=torch.bool), diagonal=1)
        s.masked_fill_(mask, float("-inf"))
    p = torch.softmax(s, dim=-1)
    return p @ v


# 准备测试 tensor
torch.manual_seed(0)
B, H, N, D = 2, 4, 256, 64
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32

q = torch.randn(B, H, N, D, device=device, dtype=dtype)
k = torch.randn(B, H, N, D, device=device, dtype=dtype)
v = torch.randn(B, H, N, D, device=device, dtype=dtype)

out_naive = naive_attention(q, k, v, causal=True)
out_sdpa  = F.scaled_dot_product_attention(q, k, v, is_causal=True)

print("Naive vs PyTorch SDPA:")
print(f"  max diff: {(out_naive - out_sdpa).abs().max().item():.2e}")
print(f"  allclose(rtol=1e-2, atol=1e-2): {torch.allclose(out_naive, out_sdpa, rtol=1e-2, atol=1e-2)}")


## 4. Triton Kernel:Mini FlashAttention2

下面这个 kernel **直接对照官方 Triton tutorial `fused-attention.py`**,做了三点简化让初学者好看:
- 只做 **forward**(backward 涉及重算 + 倒推 softmax 微分,复杂 5x)
- 只支持 **causal mask**(off-diagonal block 直接跳过)
- 假设 head_dim 是 `BLOCK_DMODEL` 的整数倍,常见 64 或 128

每个 **program(grid 上的一个工作单元)**负责一个 `(batch, head, BLOCK_M 行 Q)` 的输出块。


In [ ]:
if HAS_TRITON:
    import triton
    import triton.language as tl

    @triton.jit
    def _flash_attn_fwd_kernel(
        Q, K, V, sm_scale, Out,
        # 步长
        stride_qb, stride_qh, stride_qm, stride_qd,
        stride_kb, stride_kh, stride_kn, stride_kd,
        stride_vb, stride_vh, stride_vn, stride_vd,
        stride_ob, stride_oh, stride_om, stride_od,
        # 形状
        N_CTX,
        # 编译期常量
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
        BLOCK_DMODEL: tl.constexpr,
        IS_CAUSAL: tl.constexpr,
    ):
        # 当前 program 负责的 (batch, head, q_block) 三元组
        pid_m = tl.program_id(0)          # 沿 Q 的 BLOCK_M 维度
        pid_bh = tl.program_id(1)         # batch * head 一起编号
        n_heads = tl.num_programs(2)       # placeholder, not used here
        batch_id = pid_bh // tl.cdiv(N_CTX, BLOCK_M)  # 不,简化:用 pid_bh 直接是 batch*head 索引

        # 简单起见:让 grid = (M_blocks, B*H);pid_bh 直接是 batch*head 平铺
        # 计算 Q/K/V 在 batch*head 维度的 base offset
        # 假定 stride 是 contiguous (B, H, N, D)
        bh = pid_bh
        # 用整数 div / mod 拆 batch 和 head
        # 但这里更简单:Q/K/V 的 (B, H) 维度合并步长 = stride_qb / H?
        # —— 实战里把 (B, H) 折叠成一维传 stride 更干净;教学就显式拆
        # 简化做法:外层调用前 reshape 成 (B*H, N, D),只用 B*H 一维
        # 见下文 wrapper

        # 这里假设输入已经是 (B*H, N, D),所以 stride_qh 实际就是 stride_qb 的"行 stride"
        # 简化变量:
        q_ptr = Q + bh * stride_qh + pid_m * BLOCK_M * stride_qm
        k_ptr = K + bh * stride_kh
        v_ptr = V + bh * stride_vh
        o_ptr = Out + bh * stride_oh + pid_m * BLOCK_M * stride_om

        # 载入 Q 块:(BLOCK_M, BLOCK_DMODEL)
        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_d = tl.arange(0, BLOCK_DMODEL)
        q_block = tl.load(
            q_ptr + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd,
            mask = offs_m[:, None] < N_CTX,
            other = 0.0,
        )

        # online softmax 状态
        m_i = tl.full((BLOCK_M,), -float("inf"), dtype=tl.float32)
        l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)
        acc = tl.zeros((BLOCK_M, BLOCK_DMODEL), dtype=tl.float32)

        # 沿 K/V 块遍历
        # 因果 mask: 只看 j_block * BLOCK_N <= i_block_end
        end_n = (pid_m + 1) * BLOCK_M if IS_CAUSAL else N_CTX
        for start_n in range(0, end_n, BLOCK_N):
            offs_n = start_n + tl.arange(0, BLOCK_N)
            k_block = tl.load(
                k_ptr + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd,
                mask = offs_n[:, None] < N_CTX,
                other = 0.0,
            )
            v_block = tl.load(
                v_ptr + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd,
                mask = offs_n[:, None] < N_CTX,
                other = 0.0,
            )

            # qk^T: (BLOCK_M, BLOCK_N)
            qk = tl.dot(q_block, tl.trans(k_block))
            qk = qk * sm_scale

            # causal mask within block
            if IS_CAUSAL:
                causal_mask = offs_m[:, None] >= offs_n[None, :]
                qk = tl.where(causal_mask, qk, -float("inf"))

            # online softmax 三件套
            m_ij = tl.max(qk, axis=1)              # this block's max
            m_new = tl.maximum(m_i, m_ij)
            alpha = tl.exp(m_i - m_new)            # 旧 accumulator 的校正系数
            p = tl.exp(qk - m_new[:, None])
            l_ij = tl.sum(p, axis=1)
            l_i = alpha * l_i + l_ij

            # 校正旧 acc 并叠加新贡献
            acc = acc * alpha[:, None] + tl.dot(p.to(v_block.dtype), v_block)
            m_i = m_new

        # 最后归一化
        acc = acc / l_i[:, None]

        # 写回输出
        tl.store(
            o_ptr + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od,
            acc.to(Out.dtype.element_ty),
            mask = offs_m[:, None] < N_CTX,
        )


    def mini_flash_attn(q, k, v, causal=True):
        """q/k/v: (B, H, N, D);返回 (B, H, N, D)。"""
        B, H, N, D = q.shape
        assert q.is_cuda, "Triton 只能在 CUDA 上跑"
        assert D in (32, 64, 128), "BLOCK_DMODEL 必须是 32/64/128"

        # 折叠 (B, H) 到一维,简化 stride
        q_ = q.reshape(B * H, N, D).contiguous()
        k_ = k.reshape(B * H, N, D).contiguous()
        v_ = v.reshape(B * H, N, D).contiguous()
        o_ = torch.empty_like(q_)

        BLOCK_M, BLOCK_N = 64, 64
        sm_scale = 1.0 / (D ** 0.5)

        grid = (triton.cdiv(N, BLOCK_M), B * H)
        _flash_attn_fwd_kernel[grid](
            q_, k_, v_, sm_scale, o_,
            q_.stride(0), q_.stride(0), q_.stride(1), q_.stride(2),  # stride_qb, stride_qh, ...
            k_.stride(0), k_.stride(0), k_.stride(1), k_.stride(2),
            v_.stride(0), v_.stride(0), v_.stride(1), v_.stride(2),
            o_.stride(0), o_.stride(0), o_.stride(1), o_.stride(2),
            N_CTX = N,
            BLOCK_M = BLOCK_M,
            BLOCK_N = BLOCK_N,
            BLOCK_DMODEL = D,
            IS_CAUSAL = causal,
            num_warps = 4,
            num_stages = 2,
        )
        return o_.reshape(B, H, N, D)

    print("Triton kernel 已编译,准备 benchmark")
else:
    print("跳过 Triton 实现 —— Windows 用户可继续看 naive / SDPA 对比")


## 5. 数值校验:Triton vs SDPA


In [ ]:
if HAS_TRITON and device == "cuda":
    out_triton = mini_flash_attn(q, k, v, causal=True)
    out_sdpa   = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    max_diff = (out_triton - out_sdpa).abs().max().item()
    mean_diff = (out_triton - out_sdpa).abs().mean().item()
    ok = torch.allclose(out_triton, out_sdpa, rtol=1e-2, atol=1e-2)

    print(f"Mini-Triton vs PyTorch SDPA:")
    print(f"  max diff:  {max_diff:.2e}")
    print(f"  mean diff: {mean_diff:.2e}")
    print(f"  allclose(rtol=1e-2): {ok}")
    assert ok, "数值偏差太大,检查 mask / scale / softmax"
    print("\n[OK] 数值通过")
else:
    print("无 GPU/Triton,跳过")


## 6. 吞吐 benchmark:naive / SDPA / mini-Triton


In [ ]:
import time

def bench(fn, *args, n_warmup=5, n_iter=30):
    """返回单次 ms。仅 CUDA。"""
    if device == "cuda":
        for _ in range(n_warmup):
            fn(*args)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_iter):
            fn(*args)
        torch.cuda.synchronize()
        return (time.time() - t0) / n_iter * 1000
    else:
        for _ in range(n_warmup):
            fn(*args)
        t0 = time.time()
        for _ in range(n_iter):
            fn(*args)
        return (time.time() - t0) / n_iter * 1000


SEQ_LENS = [256, 512, 1024, 2048]
results = []
for N_test in SEQ_LENS:
    q_b = torch.randn(B, H, N_test, D, device=device, dtype=dtype)
    k_b = torch.randn(B, H, N_test, D, device=device, dtype=dtype)
    v_b = torch.randn(B, H, N_test, D, device=device, dtype=dtype)

    t_naive  = bench(lambda: naive_attention(q_b, k_b, v_b, causal=True))
    t_sdpa   = bench(lambda: F.scaled_dot_product_attention(q_b, k_b, v_b, is_causal=True))
    t_triton = bench(lambda: mini_flash_attn(q_b, k_b, v_b, causal=True)) if HAS_TRITON and device == "cuda" else None

    results.append((N_test, t_naive, t_sdpa, t_triton))
    triton_str = f"{t_triton:7.2f} ms" if t_triton else "    n/a"
    print(f"N={N_test:5d}  naive {t_naive:7.2f} ms  |  sdpa {t_sdpa:7.2f} ms  |  mini-triton {triton_str}")


In [ ]:
import matplotlib.pyplot as plt

ns = [r[0] for r in results]
plt.figure(figsize=(8, 3.5))
plt.plot(ns, [r[1] for r in results], "-o", label="naive (O(N²) HBM)")
plt.plot(ns, [r[2] for r in results], "-s", label="PyTorch SDPA (FA backend)")
if HAS_TRITON and device == "cuda":
    plt.plot(ns, [r[3] for r in results], "-^", label="mini-Triton FA2")
plt.xlabel("seq length"); plt.ylabel("ms / forward")
plt.title(f"Attention forward, B={B} H={H} D={D}, {dtype}, causal")
plt.yscale("log"); plt.legend(); plt.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()


## 7. 踩坑记录

- **Online softmax 校正一定要 `α = exp(m_old - m_new)`,不是 `exp(m_new - m_old)`**:符号搞反 acc 直接爆。证明:`acc_new = exp(m_block - m_new) * V + exp(m_old - m_new) * acc_old`,前一项把新块按新 max 归一,后一项把旧 acc 从"按 m_old 归一"重新归到"按 m_new 归一",所以乘 `exp(m_old - m_new)`。
- **Triton `tl.dot` 要求形状 ≥ (16, 16)**:`BLOCK_M / BLOCK_N / BLOCK_DMODEL` 都不能小于 16。
- **Mask 边界 `offs_m[:, None] >= offs_n[None, :]`**:`>=` 是因果(可以看自己 + 历史),`>` 会漏看自己位置,output 完全错。
- **`tl.where(mask, x, -inf)` 比 `tl.where(mask, x, -1e9)` 稳**:`-1e9` 经 exp 后仍是非零浮点,会污染 softmax;`-inf` exp 后严格 0。
- **causal mask 下 `end_n = (pid_m + 1) * BLOCK_M`** 才能正确跳过对角线之上的 K 块,这是 v2 比 v1 提速一大部分的来源(v1 跑全列,FA2 跑下三角)。
- **`num_warps` / `num_stages` 是性能调参不是正确性**:Hopper 用 num_warps=8;Ampere 用 4。调错只是慢,不会错。

## 8. 延伸阅读

- 论文:`/paper fetch 2205.14135` (FlashAttention v1,Dao 2022)
- 论文:`/paper fetch 2307.08691` (FlashAttention v2,Dao 2023)
- 论文:`/paper fetch 2407.08608` (FlashAttention v3,Hopper FP8)
- [Triton 官方 tutorial: Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html)
- [CS336 A2 Systems](https://stanford-cs336.github.io/spring2025/) —— 同款 mini FA + DDP 作业
- [[Slipbox/flashattention-v1-v2-v3]]
- [[Slipbox/online-softmax]]
